In [1]:
# Task 2: Calculate Scale and Zero Point

## Objective

#The objective of this task is to calculate the quantization parameters (scale and zero point) required for affine INT8 quantization.

#Unlike Task 1, where the scale and zero point were provided, in this task they are calculated from the input tensor. The quantized tensor is then dequantized back to floating-point values, and the quantization error is measured.

#Special attention is given to handling edge cases such as:
#- Constant tensors
#- Very small floating-point values
#- Tensors containing only positive values
#- Tensors containing only negative values

import numpy as np

 # Tensor 1: Mixed positive and negative values
t1 = np.array([-1.5, -0.8, 0.0, 0.9, 2.3], dtype=np.float32)

# Tensor 2: Only positive values
t2 = np.array([0.1, 0.5, 1.2, 2.0, 3.5], dtype=np.float32)

# Tensor 3: Only negative values
t3 = np.array([-3.0, -2.1, -1.4, -0.6, -0.1], dtype=np.float32)

# Tensor 4: Constant tensor
t4 = np.array([5.0, 5.0, 5.0], dtype=np.float32)

# Tensor 5: Very small values
t5 = np.array([1e-9, 2e-9, -1e-9], dtype=np.float32)

tensors = {
    "Tensor 1": t1,
    "Tensor 2": t2,
    "Tensor 3": t3,
    "Tensor 4": t4,
    "Tensor 5": t5
}

def calculate_scale_zero_point(tensor, q_min=-128, q_max=127):
    """
    Calculate scale and zero point for affine INT8 quantization.
    Handles constant tensors and very small ranges.
    """

    x_min = np.min(tensor)
    x_max = np.max(tensor)

    # Handle constant tensor
    if x_min == x_max:
        scale = 1.0
        zero_point = 0
        return scale, zero_point

    # Calculate scale
    scale = (x_max - x_min) / (q_max - q_min)

    # Handle extremely small scale
    if scale < 1e-12:
        scale = 1e-12

    # Calculate zero point
    zero_point = round(q_min - (x_min / scale))

    # Clip zero point
    zero_point = int(np.clip(zero_point, q_min, q_max))

    return scale, zero_point

def quantize_tensor(tensor, scale, zero_point):
    quantized = np.round(tensor / scale) + zero_point
    quantized = np.clip(quantized, -128, 127)
    return quantized.astype(np.int8)

def dequantize_tensor(quantized_tensor, scale, zero_point):
    return (quantized_tensor.astype(np.float32) - zero_point) * scale

for name, tensor in tensors.items():

    print("=" * 60)
    print(name)

    scale, zero_point = calculate_scale_zero_point(tensor)

    quantized = quantize_tensor(tensor, scale, zero_point)

    dequantized = dequantize_tensor(quantized, scale, zero_point)

    mae = np.mean(np.abs(tensor - dequantized))

    print("Tensor Values:")
    print(tensor)

    print("\nTensor Min:", np.min(tensor))
    print("Tensor Max:", np.max(tensor))

    print("\nScale:", scale)
    print("Zero Point:", zero_point)

    print("\nQuantized Tensor:")
    print(quantized)

    print("\nDequantized Tensor:")
    print(dequantized)

    print("\nMean Absolute Error (MAE):", mae)
    print()

# Observations

### Tensor 1 (Mixed Values)
#- The calculated scale and zero point allow both positive and negative values to be represented.
#- The reconstruction error is very small.

### Tensor 2 (Only Positive Values)
#- Since all values are positive, the zero point shifts toward the lower end of the INT8 range.
#- This helps represent the positive values more accurately.

### Tensor 3 (Only Negative Values)
#- Since all values are negative, the zero point shifts toward the upper end of the INT8 range.
#- The dequantized values remain very close to the original tensor.

### Tensor 4 (Constant Tensor)
#- All elements have the same value.
#- A scale of 1.0 is used to avoid division by zero.
#- No significant quantization error occurs.

### Tensor 5 (Very Small Values)
#- The calculated scale is extremely small to preserve the tiny value range.
#- The reconstructed values remain close to the originals, although small numerical differences may appear due to floating-point precision.

# Conclusion

#In this task, the scale and zero point were calculated automatically for different types of tensors. The implementation successfully handled various edge cases, including constant tensors and tensors with very small values. The results show that proper calculation of quantization parameters helps minimize reconstruction error and is an essential step in affine INT8 quantization.

Tensor 1
Tensor Values:
[-1.5 -0.8  0.   0.9  2.3]

Tensor Min: -1.5
Tensor Max: 2.3

Scale: 0.01490196
Zero Point: -27

Quantized Tensor:
[-128  -81  -27   33  127]

Dequantized Tensor:
[-1.505098   -0.80470586  0.          0.8941176   2.2949018 ]

Mean Absolute Error (MAE): 0.004156864

Tensor 2
Tensor Values:
[0.1 0.5 1.2 2.  3.5]

Tensor Min: 0.1
Tensor Max: 3.5

Scale: 0.013333334
Zero Point: -128

Quantized Tensor:
[-120  -90  -38   22  127]

Dequantized Tensor:
[0.10666667 0.50666666 1.2        2.         3.4       ]

Mean Absolute Error (MAE): 0.022666646

Tensor 3
Tensor Values:
[-3.  -2.1 -1.4 -0.6 -0.1]

Tensor Min: -3.0
Tensor Max: -0.1

Scale: 0.011372549
Zero Point: 127

Quantized Tensor:
[-128  -58    4   74  118]

Dequantized Tensor:
[-2.9        -2.1039217  -1.3988236  -0.6027451  -0.10235295]

Mean Absolute Error (MAE): 0.022039209

Tensor 4
Tensor Values:
[5. 5. 5.]

Tensor Min: 5.0
Tensor Max: 5.0

Scale: 1.0
Zero Point: 0

Quantized Tensor:
[5 5 5]

Dequantized Ten